In [ ]:
import pandas as pd
from pprint import pprint
import random


In [ ]:
# Constants
ALLOWED_NUMBER_OF_COURSES = 3

# PreReqs
prereq_dict = {
    "DATM507": ["ITOP501"],
    "DATM508": ["ITOP501"],
    "ITOP601": ["ITOP501"],
    "PROG602": ["DATM507"],
    "DATM603": ["DATM506", "DATM507"],
    "DATM604": ["PROG504", "DATM506"],
    "DATM605": ["PROG504"],
    "SYSP606": ["ITOP501"],
    "ITMG607": ["DATM508", "ITOP601"],
    "PROF608": ["SYSP606"],
    "PRMG701": ["PROF502", "PROG602", "SYSP606"],
    "HCEV702": ["PROF505", "SYSP606", "PROG602"],
    "DATM703": ["DATM603", "DATM604", "ITMG607"],
    "NTAS704": ["SCTY503", "ITMG607"],
    "PROF710": ["PROF709"],
    "ITOP721": ["DATM604"],
    "PROG722": ["PROG504", "DATM605"],
    "ITOP723": ["ITMG607"],
    "IRMG724": ["PROF505", "PROF608", "PRMG701"]
}




In [ ]:
def update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo):
    dict_subject_stats = {}
    for course_id in list_courses:
        dict_subject_stats[course_id] = df_with_prereqinfo[course_id].value_counts().to_dict()
        tot_eligible_students = 0
        for iter_key in keys_to_visit_for_elgible_students:
            if iter_key in dict_subject_stats[course_id]:
                tot_eligible_students += dict_subject_stats[course_id][iter_key]
        # Add back to dictionary
        dict_subject_stats[course_id]['tot_eligible_students'] = tot_eligible_students

    return dict_subject_stats

def func_filter_eligible_students(list_eligible_students, dict_trackCourseAssignments):
    filtered_list = []
    for student_id in list_eligible_students:
        if dict_trackCourseAssignments[student_id]["freeze_assignment"] is False:
            filtered_list.append(student_id)
    return filtered_list

# npr: No PreReq Required (no pre-req required for this course)
# pum: Prereq unmet (Prereq not passed)
# met: Prereq met 
def check_course_eligibility(course_id, prereq_dict, dict_student):
    if course_id in prereq_dict:
        eligible_course_list = prereq_dict[course_id]
        for iter_eligible_course in eligible_course_list:
            status = dict_student[iter_eligible_course]
            if status != "p":
                return "pum"
        return "met"
    else:
        return "npr"

def mark_eligible_courses(df_dict, prereq_dict):
    df_dict_with_prereq_status = {}
    for iter_student in df_dict:
        print(f"Student-ID: {iter_student}")
        dict_student = df_dict[iter_student]
        df_dict_with_prereq_status[iter_student] = {}
        for iter_sub in dict_student:
            prereq_status = "already_passed" if dict_student[iter_sub] == "p" else check_course_eligibility(iter_sub, prereq_dict, dict_student)
            print(f"iter_student: {iter_student}, iter_sub:{iter_sub}, status:{dict_student[iter_sub]}, prereq_status: {prereq_status}")
            # Fill the dictionary
            df_dict_with_prereq_status[iter_student][iter_sub] = prereq_status
    return df_dict_with_prereq_status


def func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected):
    # Filter df
    df_filtered = df_with_prereqinfo.loc[(df_with_prereqinfo[course_id] == "npr") | (df_with_prereqinfo[course_id] == "met")][course_id]
    list_eligible_students = df_filtered.index.to_list()
    list_eligible_students = func_filter_eligible_students(list_eligible_students, dict_trackCourseAssignments)
    
    if len(list_eligible_students) == 0:
        print("Eligible students not found")
    elif no_of_students_selected > len(list_eligible_students): 
        print(f"Can not select {no_of_students_selected} from {len(list_eligible_students)} eligible students. Reduce your selection.")
    else:
        list_sampled_students = random.sample(list_eligible_students, no_of_students_selected)
        print(list_sampled_students, len(list_sampled_students))
        # Modify the entries of df_with_prereqinfo
        for iter_eligible_studentid in list_sampled_students:
            df_with_prereqinfo.at[iter_eligible_studentid, course_id] = "assigned"
            # Update df_dict_with_prereq_status and dict_trackCourseAssignments 
            if dict_trackCourseAssignments[iter_eligible_studentid]["freeze_assignment"] is False:
                df_dict_with_prereq_status[iter_eligible_studentid][course_id] = "assigned"
                dict_trackCourseAssignments[iter_eligible_studentid]["assigned"] += 1
                dict_trackCourseAssignments[iter_eligible_studentid]["limit"] = ALLOWED_NUMBER_OF_COURSES


                if dict_trackCourseAssignments[iter_eligible_studentid]["assigned"] == dict_trackCourseAssignments[iter_eligible_studentid]["limit"]:
                    dict_trackCourseAssignments[iter_eligible_studentid]["freeze_assignment"] = True
                    # Lock courses
                    list_courses = df_with_prereqinfo.columns.to_list()
                    for iter_course_ids in list_courses:
                        # Lock the entry
                        df_with_prereqinfo.at[iter_eligible_studentid, iter_course_ids] = df_with_prereqinfo.at[iter_eligible_studentid, iter_course_ids] + "-LOCKED"


    return df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status
    

def print_eligibility(dict_subject_stats, list_courses):
    key_to_visit = "tot_eligible_students"
    for course_id in list_courses:
        print(f"Eligible students in {course_id}: {dict_subject_stats[course_id][key_to_visit]}")

In [ ]:
df = pd.read_csv("mastersheet_trimmed.csv")
df = df.fillna("x")


In [ ]:
df = df.set_index('Studentid')
df_dict = df.to_dict(orient='index')

In [ ]:
df_dict.keys()

In [ ]:
df_dict_with_prereq_status = mark_eligible_courses(df_dict, prereq_dict)

In [ ]:
df_with_prereqinfo = pd.DataFrame.from_dict(df_dict_with_prereq_status, orient='index')

In [ ]:
df_with_prereqinfo

In [ ]:
# Prepare a dictionary for all students to track how many courses are assigned to each student
dict_trackCourseAssignments = {}
for iter_student in df_dict:
    dict_trackCourseAssignments[iter_student] = {'assigned': 0, 'limit': ALLOWED_NUMBER_OF_COURSES, 'freeze_assignment': False}
print(dict_trackCourseAssignments)

In [ ]:
# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print(dict_subject_stats)


# Assigning Subjects to Eligible Students
1. Select Subject and find how many students can enrol in it (use dict_subject_stats)
2. Get list of eligible students for the above subject (get_eligible_students(str: subject_id) -> List)
3. Use a parameter and select students randomly from that list
4. Update df_with_prereqinfo and modify cell value by "assigned"


In [ ]:
course_id = 'DATM506'
no_of_students_selected = 25

df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)


In [ ]:
df_with_prereqinfo.to_csv("check.csv")

In [ ]:
course_id = 'ITOP601'
no_of_students_selected = 25
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)


In [ ]:
df_with_prereqinfo.to_csv("check.csv")

In [ ]:
course_id = 'DATM604'
no_of_students_selected = 25
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)

In [ ]:
course_id = 'DATM508'
no_of_students_selected = 47
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)

In [ ]:
course_id = 'DATM605'
no_of_students_selected = 141
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)

In [ ]:
course_id = 'ITMG607'
no_of_students_selected = 23
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)

In [ ]:
df_with_prereqinfo.to_csv("check.csv")